# 03 – Belastung: TRIMP, Fitness, Ermüdung und Form

Ein einfaches Belastungsmodell nach Banister, berechnet über **alle Sportarten**:

- **TRIMP** (Trainingsimpuls) pro Aktivität: Dauer × Intensität. Die Intensität ist die Herzfrequenzreserve HRr = (HF − HF_Ruhe) ÷ (HF_max − HF_Ruhe), exponentiell gewichtet, damit harte Minuten überproportional zählen.
- **CTL** („Fitness“): gleitender, exponentiell gewichteter Mittelwert der täglichen TRIMP über etwa 42 Tage.
- **ATL** („Ermüdung“): dasselbe über 7 Tage. Reagiert schnell.
- **TSB** („Form“) = CTL − ATL des Vortags, also die Form am Morgen vor dem Training. Positiv heißt frisch, negativ ermüdet.

**Parameter** (`config.toml`, bestätigt 27.10.2026): HF_max = 183 (höchster gemessener Wert), Konstanten für männlich k₁ = 0,64, k₂ = 1,92. HF_Ruhe ist der Ruhepuls des jeweiligen Tages aus `daily`. Formeln in `metrics.py`, Prüfungen in `checks.py`.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from myfitness import explore, metrics, plots
from myfitness.config import PROCESSED

activities = pl.read_parquet(PROCESSED / "activities.parquet")
activity_metrics = pl.read_parquet(PROCESSED / "activity_metrics.parquet")

## Alle Sportarten

In [ ]:
load_all = metrics.training_load(activity_metrics, activities)
plots.plot_training_load(load_all, "Belastung – alle Sportarten")
plt.show()

**Einschwingphase:** Das Modell startet am ersten Datentag bei 0. In den ersten etwa 6 Wochen ist die CTL deshalb zu niedrig. Starke TSB-Ausschläge in dieser Zeit sind vor allem ein Artefakt dieses Starts.

In [ ]:
# Where does the load come from?
(
    activity_metrics.join(activities.select("activity_id", "sport"), on="activity_id")
    .group_by("sport")
    .agg(activities=pl.len(), trimp_total=pl.col("trimp").sum(), trimp_mean=pl.col("trimp").mean())
    .with_columns(share_pct=pl.col("trimp_total") / pl.col("trimp_total").sum() * 100)
    .sort("trimp_total", descending=True)
)

## Nach Sportart filtern

`SPORTS` auf eine Liste setzen, z. B. `["running"]` oder `["running", "cycling"]`. Mit `None` gehen alle Sportarten ein.

In [ ]:
SPORTS = ["running"]

load_selected = metrics.training_load(activity_metrics, activities, sports=SPORTS)
plots.plot_training_load(load_selected, f"Belastung – {', '.join(SPORTS)}")
plt.show()

## Prüfung: Senken Ruhetage die Ermüdung?

An Tagen ohne Training muss die ATL fallen, und zwar um 1/7 ihres Werts. `diff()` zeigt die Veränderung zum Vortag.

In [ ]:
(
    load_all.with_columns(atl_change=pl.col("atl").diff(), rest_day=pl.col("trimp") == 0)
    .group_by("rest_day")
    .agg(days=pl.len(), mean_atl_change=pl.col("atl_change").mean(), share_atl_falls=(pl.col("atl_change") < 0).mean())
    .sort("rest_day")
)

## Zurück zu Leitfrage 2: War es nur der Ruhetag?

In `02_korrelationen` wird geprüft, ob gute Erholungswerte am **Vortag** mit einer besseren Tagesform einhergehen (`ef_vs_trend`). Falls ja, liegt eine Erklärung nahe: Gute Vortagswerte gibt es nach einem Ruhetag, und dann ist eigentlich die Pause der Grund.

Als Ruhetag zählt hier ein Vortag mit weniger als 10 TRIMP, ein Spaziergang ist also noch kein Training.

**Lesehilfe:** Sind `mean_sleep_score_lag1` und `mean_resting_hr_lag1` nach Ruhetagen besser als nach Trainingstagen? Nur dann kommt der Ruhetag als Erklärung überhaupt in Frage.

In [ ]:
runs = explore.with_training_load(explore.run_table()).filter(pl.col("ef").is_not_null())

runs.group_by("rest_day_before").agg(
    runs=pl.len(),
    mean_ef_vs_trend=pl.col("ef_vs_trend").mean(),
    mean_sleep_score_lag1=pl.col("sleep_score_lag1").mean(),
    mean_resting_hr_lag1=pl.col("resting_hr_lag1").mean(),
).sort("rest_day_before")

Hält der Zusammenhang **innerhalb** beider Gruppen? Wenn ja, erklärt der Ruhetag ihn nicht.

In [ ]:
recovery_values = ["sleep_score_lag1", "resting_hr_lag1", "hrv_overnight_avg_lag1", "body_battery_high_lag1"]
rows = []
for (rest_day,), group in runs.group_by("rest_day_before"):
    for value in recovery_values:
        rows.append({"rest_day_before": rest_day, **explore.correlate(group, value, "ef_vs_trend")})
pl.DataFrame(rows).pivot(on="rest_day_before", index="x", values="spearman").rename(
    {"true": "nach Ruhetag", "false": "nach Trainingstag"}
)

**Lesehilfe:** Hat der Zusammenhang in beiden Spalten dasselbe Vorzeichen und eine ähnliche Größe, besteht er unabhängig vom Ruhetag, und die Ruhetag-Erklärung trägt nicht. Tritt er nur nach Ruhetagen auf, spricht das eher für die Pause als Grund. Die Gruppen sind kleiner als die Gesamtzahl der Läufe, auf `runs` oben achten.

Und die Form laut Modell?

In [ ]:
explore.investigate(runs, "tsb", "ef_vs_trend")

**Lesehilfe:** Ein positives ρ hieße, dass Läufe an frischen Tagen (TSB > 0) effizienter sind. Liegt ρ nahe 0, bildet das Modell die Belastung ab, sagt aber nicht vorher, wie effizient ein einzelner Lauf wird.

## Grenzen der Aussage

- **Einschwingphase:** CTL/ATL beginnen bei 0. Der Anfang des Zeitraums ist nicht aussagekräftig.
- **TRIMP misst nur die Herzfrequenz.** Krafttraining, kurze Intervalle und Atemübungen erzeugen wenig HF-Belastung und werden unterschätzt. Gehen zählt dagegen voll mit.
- **HF_max** ist der höchste gemessene Wert. Ist der echte Maximalpuls höher, wird jede Einheit etwas überschätzt.
- **Ruhetag-Schwelle** (10 TRIMP) und die Zeitkonstanten (42/7 Tage) sind Konventionen, keine Messgrößen.
- n = 1, Beobachtungsdaten. Die Ergebnisse sind Hypothesen.